# ToolRL Reproduction Notebook

Provisions a Chameleon Cloud [cloud computing platform for research] GPU instance, sets up Docker [software container system], and runs one training case end-to-end.

**Target:** GRPO [Group Relative Policy Optimization -- RL algorithm that doesn't need a separate critic model] Cold Start [trained straight from the base model, no supervised fine-tuning first], Qwen2.5-1.5B [1.5 billion parameter language model by Alibaba]

**Paper result (API-Bank [the paper's own 597-question tool-call benchmark]):** 63.15% | **Our result:** 58.96%

**Wall time:** ~3.5 hours on 4x H100 [NVIDIA GPU]

**Before running:**
- Chameleon Cloud account with an active allocation at kvm.tacc.chameleoncloud.org
- SSH keypair [public/private key pair for secure remote login] registered in the KVM@TACC dashboard
- `pip install python-chi`

---

## Table of Contents

| Section | What it does |
|---------|-------------|
| [0. Configuration](#0-configuration) | Set your project name, keypair, GPU flavor |
| [1. Provision Instance](#1-provision-the-instance) | Create lease + launch VM. Cell 1c creates a Cinder volume [persistent disk] |
| [2. Configure Security Groups and Attach Storage](#2-configure-security-groups-and-attach-storage) | Open ports, attach + mount the 512 GB Cinder volume at `/mnt/block` |
| [3. Install Docker](#3-install-docker-and-nvidia-container-toolkit) | Install Docker + NVIDIA container support |
| [4. Move Storage & Clone Repo](#4-move-docker--containerd-storage-to-block-volume-clone-repo-create-named-volumes) | Move Docker + containerd to `/mnt/block`, clone repo, create named volumes |
| [5. Build Docker Image](#5-build-the-docker-image) | Build the training container (~15-20 min) |
| [6. Start Containers](#6-start-containers) | Launch verl [training container] + mlflow [experiment tracker] |
| [7. Prepare Dataset](#7-prepare-the-dataset) | Convert raw JSON to parquet [columnar data format the trainer needs] |
| [8. Download Base Model](#8-download-the-base-model) | Pull Qwen2.5-1.5B from HuggingFace [model hosting platform] |
| [9. Run Training](#9-run-training) | Run GRPO demo (1 step, ~4-5 min) or full 15-epoch reproduction |
| [10. Evaluate on API-Bank](#10-evaluate-on-api-bank) | Run 597-question benchmark, get accuracy score |
| [11. Check Disk & Checkpoints](#11-check-disk--clean-old-checkpoints) | Monitor disk, kill jobs, clean old checkpoints [saved model weights] |
| [12. Cleanup](#12-cleanup) | Check lease, stop containers, delete server/lease/volume |
| [(Optional) Resume from Volume](#optional-resume-from-existing-cinder-volume) | Boot new VM from existing Cinder volume -- skips all setup |

---

## Which path are you on?

**First time setup (no existing volume):**
Run sections in order: 0 -> 1 -> 2 -> 3 -> 4 -> 5 -> 6 -> 7 -> 8 -> 9 -> 10. When done: 11 (Check Disk & Checkpoints) -> 12 (Cleanup).

**Resuming from an existing Cinder volume [persistent disk that survived your last reservation]:**
Your Docker image, model weights, dataset, and any previous checkpoints are already on the volume -- no reinstall needed. Run the [(Optional) Resume from Existing Cinder Volume](#optional-resume-from-existing-cinder-volume) cell at the bottom first (boots a new VM from your existing volume), then pick up at: 2 (Connect via SSH) -> 6 (Start Containers) -> 11 (Check Disk & Clean Old Checkpoints) -> 9 (Run Training) -> 10 (Evaluate on API-Bank). When done: 12 (Cleanup).

## 0. Configuration

In [ ]:
# --- Configuration ---
# All resources (lease, server, volume) will be named with this prefix.
PROJECT_ID  = "projXX"           # change to a short identifier of your choice (e.g. your netID)
LEASE_HOURS = 12
FLAVOR      = "gpu.h100.4x"      # change to "g1.h100.pci.1" (full H100) or "g1.h100.vgpu.1g.12gb" (vGPU slice) per availability
IMAGE_NAME  = "CC-Ubuntu24.04-CUDA"   # use "CC-Ubuntu24.04-CUDA-VGPU" if FLAVOR is a *.vgpu.* slice
N_GPUS      = 4                  # must match FLAVOR (1 for g1.h100.pci.1, 4 for gpu.h100.4x)
REPO_URL    = "https://github.com/Mario928/toolrl-verl-reproduction"

from chi import server, context, lease, network
import chi, os, datetime, time

context.version = "1.0"
context.choose_project()
context.choose_site(default="KVM@TACC")


## 1. Provision the Instance

Run the cells in order: cell 1a creates the lease, cell 1c (optional) creates the persistent Cinder block volume, cell 1b launches the GPU VM using the lease's flavor reservation. All three submits are idempotent -- safe to re-run.


In [ ]:
# --- 1a: Create lease (idempotent -- safe to re-run; reuses existing lease with same name) ---
l = lease.Lease(f"{PROJECT_ID}-toolrl-lease", duration=datetime.timedelta(hours=LEASE_HOURS))
l.add_flavor_reservation(id=chi.server.get_flavor_id(FLAVOR), amount=1)
l.submit(idempotent=True)
l.show()


### 1c. (Optional) Create a Cinder Block Volume

Skip this if you already have a volume from a previous run (check KVM@TACC dashboard > Volumes).

A Cinder volume gives you persistent large disk (512GB+) that survives instance deletion. Without it, the VM gets ~40GB which is not enough for the Docker image + model weights (~60-70GB needed).

The volume is created here, then attached + mounted at `/mnt/block` in Section 2b after the VM is up. Docker's data-root is then pointed at `/mnt/block/docker` in Section 4 so all named volumes live on the 512 GB disk.


In [ ]:
# --- 1c: Create a Cinder block volume (idempotent -- reuses existing volume with same name) ---
# Persists across reservations. 512GB is enough for the verl image + models + checkpoints.
VOLUME_NAME    = f"{PROJECT_ID}-toolrl-persist"
VOLUME_SIZE_GB = 512

cinder_client = chi.clients.cinder()
existing = [v for v in cinder_client.volumes.list() if v.name == VOLUME_NAME]
if existing:
    volume = existing[0]
    print(f"Found existing volume: {volume.name} (status: {volume.status})")
else:
    volume = cinder_client.volumes.create(name=VOLUME_NAME, size=VOLUME_SIZE_GB)
    print(f"Created new volume: {volume.name}")

# Wait for available
volume = cinder_client.volumes.get(volume.id)
while volume.status not in ["available", "in-use"]:
    print(f"Volume status: {volume.status}, waiting...")
    time.sleep(5)
    volume = cinder_client.volumes.get(volume.id)
print(f"Volume status: {volume.status}")


In [ ]:
# --- 1b: Launch instance (idempotent -- safe to re-run; reuses existing server with same name) ---
s = server.Server(
    f"{PROJECT_ID}-toolrl-server",
    image_name=IMAGE_NAME,
    flavor_name=l.get_reserved_flavors()[0].name,
)
s.submit(idempotent=True, show="text")
s.associate_floating_ip()
s.refresh()
s.check_connectivity()
s.show(type="widget")
floating_ip = s.get_floating_ip()
print(f"Floating IP: {floating_ip}")


## 2. Configure Security Groups and Attach Storage

Open the required ports on the instance and attach + mount the Cinder volume so the
large Docker images and model checkpoints land on the 512 GB block storage instead of
the small root disk.


In [ ]:
# --- 2a: Open required ports via security groups ---
# Idempotent -- re-running just re-applies the groups.
security_groups = [
    {"name": "allow-ssh",  "port": 22,   "description": "Enable SSH on TCP 22"},
    {"name": "allow-5000", "port": 5000, "description": "Enable MLflow UI on TCP 5000"},
    {"name": "allow-8000", "port": 8000, "description": "Enable additional service on TCP 8000"},
    {"name": "allow-8888", "port": 8888, "description": "Enable Jupyter on TCP 8888"},
]

for sg in security_groups:
    secgroup = network.SecurityGroup({
        "name": sg["name"],
        "description": sg["description"],
    })
    secgroup.add_rule(direction="ingress", protocol="tcp", port=sg["port"])
    secgroup.submit(idempotent=True)
    s.add_security_group(sg["name"])

print(f"Updated security groups: {[sg['name'] for sg in security_groups]}")


In [ ]:
# --- 2b: Attach + format + mount the Cinder volume at /mnt/block ---
volume = cinder_client.volumes.get(volume.id)
if volume.status == "available":
    s.attach_volume(volume.id)
    print(f"Attached volume {VOLUME_NAME} to server")
else:
    print(f"Volume status is {volume.status}, skipping attach (probably already attached)")

# Discover the device path dynamically (don't hardcode /dev/vdb)
volume = cinder_client.volumes.get(volume.id)
device = volume.attachments[0]["device"]   # e.g. /dev/vdb
part = f"{device}1"

s.execute(f"""
set -e
if ! sudo blkid {part} >/dev/null 2>&1; then
    echo "No filesystem on {part}; creating GPT + ext4"
    sudo parted -s {device} mklabel gpt
    sudo parted -s {device} mkpart primary ext4 0% 100%
    sudo mkfs.ext4 {part}
else
    echo "Filesystem already exists on {part}; skipping format"
fi
sudo mkdir -p /mnt/block
if ! mountpoint -q /mnt/block; then
    sudo mount {part} /mnt/block
fi
sudo chown -R cc:cc /mnt/block
""")

# Create directory where Docker's data-root will live (used in Section 3)
s.execute("sudo mkdir -p /mnt/block/docker")


## 3. Install Docker and nvidia-container-toolkit

In [ ]:
s.execute("sudo apt-get update -q")
s.execute("sudo apt-get install -y -q ca-certificates curl gnupg lsb-release")
s.execute("curl -fsSL https://download.docker.com/linux/ubuntu/gpg | sudo gpg --dearmor -o /usr/share/keyrings/docker-archive-keyring.gpg")
s.execute('echo "deb [arch=$(dpkg --print-architecture) signed-by=/usr/share/keyrings/docker-archive-keyring.gpg] https://download.docker.com/linux/ubuntu $(lsb_release -cs) stable" | sudo tee /etc/apt/sources.list.d/docker.list > /dev/null')
s.execute("sudo apt-get update -q")
s.execute("sudo apt-get install -y -q docker-ce docker-ce-cli containerd.io docker-compose-plugin")
s.execute("sudo usermod -aG docker cc")
print("Docker installed.")

In [ ]:
s.execute("curl -fsSL https://nvidia.github.io/libnvidia-container/gpgkey | sudo gpg --dearmor -o /usr/share/keyrings/nvidia-container-toolkit-keyring.gpg")
s.execute("curl -s -L https://nvidia.github.io/libnvidia-container/stable/deb/nvidia-container-toolkit.list | sed 's#deb https://#deb [signed-by=/usr/share/keyrings/nvidia-container-toolkit-keyring.gpg] https://#g' | sudo tee /etc/apt/sources.list.d/nvidia-container-toolkit.list")
s.execute("sudo apt-get update -q")
s.execute("sudo apt-get install -y -q nvidia-container-toolkit")
s.execute("sudo nvidia-ctk runtime configure --runtime=docker")
s.execute("sudo systemctl restart docker")
print("nvidia-container-toolkit installed.")

In [ ]:
stdout, _ = s.execute("sudo docker run --rm --gpus all nvidia/cuda:12.6.3-base-ubuntu24.04 nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")
print(stdout)

## 4. Move Docker + Containerd Storage to Block Volume, Clone Repo, Create Named Volumes

Before building the large verl image, point both Docker AND containerd at the 512 GB Cinder volume mounted at `/mnt/block`. Without this the verl image build fails mid-way (`no space left on device`) because containerd snapshots live on the small root disk by default.


In [ ]:
# --- Move Docker AND containerd storage to /mnt/block so the 512 GB Cinder volume
#     (not the small ~40 GB root disk) absorbs the verl image build + model weights.
#     We move BOTH because Docker's "data-root" only relocates Docker metadata; the
#     containerd snapshotter (where layers actually live during `docker build`) writes
#     to /var/lib/containerd, which is on the root disk by default. Without moving
#     containerd, `docker build` fails with "no space left on device" around step 8-10
#     (vllm + flash-attn install). ---
s.execute("sudo systemctl stop docker docker.socket containerd")

# Relocate containerd
s.execute("sudo mkdir -p /mnt/block/containerd")
s.execute("sudo rsync -aHX /var/lib/containerd/ /mnt/block/containerd/ 2>/dev/null || true")
s.execute("sudo rm -rf /var/lib/containerd")
s.execute("sudo ln -s /mnt/block/containerd /var/lib/containerd")

# Relocate docker data
s.execute("sudo mkdir -p /mnt/block/docker")
s.execute("sudo rsync -aHX /var/lib/docker/ /mnt/block/docker/ 2>/dev/null || true")
s.execute("sudo rm -rf /var/lib/docker")
s.execute("sudo ln -s /mnt/block/docker /var/lib/docker")

# MERGE data-root into daemon.json. We write a tiny Python script to a temp file
# (via bash heredoc with single-quoted EOF to suppress variable expansion), then
# run it. This avoids the shell-quote hell of trying to pass JSON literals through
# `python3 -c "..."` or `jq '...'` chains over SSH.
# IMPORTANT: nvidia-ctk wrote the nvidia runtime config to daemon.json in Section 3.
# A naive `tee > daemon.json` would wipe it; the merge below preserves it.
s.execute("""cat > /tmp/set_docker_root.py << 'EOF'
import json, os
p = '/etc/docker/daemon.json'
d = json.load(open(p)) if os.path.exists(p) else {}
d['data-root'] = '/mnt/block/docker'
json.dump(d, open(p, 'w'), indent=2)
print(open(p).read())
EOF""")
s.execute("sudo python3 /tmp/set_docker_root.py")

s.execute("sudo systemctl start containerd docker")
# Verify both nvidia runtime AND new data-root are active
s.execute("sudo docker info | grep -E 'Docker Root Dir|Runtimes'")


In [ ]:
s.execute(f"git clone {REPO_URL} /home/cc/toolrl")
for vol in ["toolrl_models", "toolrl_hf_cache", "toolrl_mlflow_data", "toolrl_datasets"]:
    s.execute(f"sudo docker volume create {vol}")
print("Done.")

## 5. Build the Docker Image

Takes ~15-20 minutes the first time.

In [ ]:
stdout, _ = s.execute("cd /home/cc/toolrl && sudo docker build -t toolrl-verl:latest . 2>&1 | tail -20")
print(stdout)

## 6. Start Containers

In [ ]:
s.execute("cd /home/cc/toolrl && sudo docker compose up -d")
stdout, _ = s.execute("sudo docker ps --format 'table {{.Names}}\t{{.Status}}'")
print(stdout)
print(f"MLflow UI [experiment tracker -- tracks loss, reward, and training metrics]: http://{floating_ip}:5000")

## 7. Prepare the Dataset

Converts the raw JSON files already in the repo into the parquet format the trainer expects. Must run before training or the trainer crashes at step 1 with `KeyError: 'reward_model'`.

In [ ]:
s.execute("sudo docker exec verl bash -c 'cd /workspace && python dataset/rlla_4k_raw/rlla.py'")
print("Dataset prepared.")

In [ ]:
s.execute(
    "sudo docker exec verl python3 -c "
    "'from huggingface_hub import snapshot_download; "
    "snapshot_download(\"Qwen/Qwen2.5-1.5B-Instruct\")'"
)
print("Model downloaded.")

## 9. Run Training

**Before running:** If you are resuming from a previous run, check disk and delete old checkpoints first (Section 11) to avoid running out of space. Each checkpoint is ~6-7 GB.

**Demo mode (active below):** 1 epoch, small batch — completes in ~10-15 min on 1x H100. Shows the full training loop working end-to-end.

**Full reproduction run:** 15 epochs, batch=512 — takes ~3.5h on 4x H100 / ~7h on 1x H100. Reaches the paper's checkpoint at `global_step_90` with 58.96% API-Bank accuracy. Swap in the commented config below to run it.

To run a different reward variant, set the reward flags (`COARSEREWARD=1`, `REFINEDREWARD=1`, etc.) before the training command.

In [ ]:
# --- Run demo training: 1 step, batch=64, ~10-15 min on 1x H100 ---
s.execute("""sudo docker exec -d -e EXPERIMENT_NAME=grpo_qwen_1.5b_demo verl bash -c 'cd /workspace && CUDA_VISIBLE_DEVICES=0 VLLM_ATTENTION_BACKEND=XFORMERS python3 -m verl.trainer.main_ppo algorithm.adv_estimator=grpo data.train_files=./dataset/rlla_4k/train_64.parquet data.val_files=./dataset/rlla_4k/test.parquet data.train_batch_size=64 data.val_batch_size=32 data.max_prompt_length=2048 data.max_response_length=1024 actor_rollout_ref.model.path=Qwen/Qwen2.5-1.5B-Instruct actor_rollout_ref.actor.optim.lr=1e-6 actor_rollout_ref.model.use_remove_padding=True actor_rollout_ref.actor.ppo_mini_batch_size=64 actor_rollout_ref.actor.use_dynamic_bsz=True actor_rollout_ref.actor.use_kl_loss=False actor_rollout_ref.actor.kl_loss_coef=0.001 actor_rollout_ref.actor.kl_loss_type=low_var_kl actor_rollout_ref.model.enable_gradient_checkpointing=True actor_rollout_ref.actor.fsdp_config.param_offload=False actor_rollout_ref.actor.fsdp_config.grad_offload=False actor_rollout_ref.actor.fsdp_config.optimizer_offload=False actor_rollout_ref.rollout.tensor_model_parallel_size=1 actor_rollout_ref.rollout.name=vllm actor_rollout_ref.rollout.gpu_memory_utilization=0.6 actor_rollout_ref.rollout.n=4 actor_rollout_ref.ref.fsdp_config.param_offload=True algorithm.kl_ctrl.kl_coef=0.001 trainer.critic_warmup=0 trainer.logger=[console,mlflow] trainer.project_name=toolrl trainer.default_local_dir=/app/models/toolrl-grpo-cold-qwen-1.5b-demo trainer.experiment_name=grpo_qwen_1.5b_demo trainer.n_gpus_per_node=1 trainer.nnodes=1 trainer.save_freq=1 trainer.test_freq=1 trainer.total_epochs=1 > /tmp/train.log 2>&1'""")
print(f"Training launched. MLflow: http://{floating_ip}:5000")

# Tail log until step:1 appears (run this in a loop or re-run manually)
import time
for _ in range(40):
    stdout, _ = s.execute("sudo docker exec verl tail -3 /tmp/train.log 2>/dev/null || echo 'starting...'")
    print(stdout.strip())
    if 'step:1' in stdout or 'Final validation' in stdout:
        print("\nTraining complete!")
        break
    time.sleep(20)

### Inspect: Dataset, GPU, Model Output

Run any of these cells during or after training to answer professor questions live.

In [ ]:
# --- Show a sample from the training dataset [the tool-call questions the model learns from] ---
stdout, _ = s.execute("""
sudo docker exec verl python3 -c "
import pandas as pd, json
df = pd.read_parquet('./dataset/rlla_4k/train.parquet')
print(f'Train size: {len(df)} rows | Columns: {list(df.columns)}')
print()
row = df.iloc[0]
# prompt is a list of chat messages
msgs = row['prompt'] if isinstance(row['prompt'], list) else json.loads(row['prompt'])
for m in msgs:
    print(f'[{m[\"role\"].upper()}]')
    print(m['content'][:300])
    print()
"
""")
print(stdout)

In [ ]:
# --- GPU utilization [how hard the GPU is working] during training ---
stdout, _ = s.execute("sudo docker exec verl nvidia-smi --query-gpu=name,utilization.gpu,memory.used,memory.total,temperature.gpu --format=csv,noheader")
print(stdout)

In [ ]:
# --- Show a raw model output sample [what the model actually generates for a tool-call question] ---
# Run after training completes. Uses the demo checkpoint.
stdout, _ = s.execute("""
sudo docker exec verl python3 -c "
from vllm import LLM, SamplingParams
import pandas as pd

llm = LLM(model='/app/models/toolrl-grpo-cold-qwen-1.5b-demo/actor/global_step_1', gpu_memory_utilization=0.6)
params = SamplingParams(temperature=0.0, max_tokens=256)

df = pd.read_parquet('./dataset/rlla_4k/test.parquet')
msgs = df.iloc[0]['prompt']
prompt = '\n'.join(f'[{m[\"role\"].upper()}] {m[\"content\"]}' for m in msgs) + '\n[ASSISTANT]'
out = llm.generate([prompt], params)[0].outputs[0].text
print('=== PROMPT (first test question) ===')
print(prompt[:500])
print()
print('=== MODEL OUTPUT ===')
print(out)
"
""")
print(stdout)

In [ ]:
# --- Show the reward function [the code that scores whether the model's tool call was correct] ---
# stdout, _ = s.execute("sudo docker exec verl cat /workspace/verl/utils/reward_score/rlla.py")
# print(stdout)

# HOW THE REWARD FUNCTION WORKS:
# --------------------------------
# 1. The model generates a response to a tool-call question.
#    Example question: "What is the weather in New York?"
#    Expected output:  <tool_call>GetWeather(location="New York")</tool_call>
#
# 2. FORMAT CHECK [did the model wrap its answer in <tool_call> tags?]
#    - Correct format  -> +1
#    - Missing tags    ->  0  (model didn't learn the output structure yet)
#
# 3. CORRECTNESS CHECK [did the model call the right API with the right parameters?]
#    - Exact match [API name + all parameters correct] -> +3
#    - Wrong API name                                  -> -3
#    - Right API, wrong parameters                     -> -3 (default) or partial credit if COARSEREWARD=1
#
# 4. FINAL SCORE = format score + correctness score
#    Range: -3 (totally wrong) to +4 (correct format AND correct answer)
#
# 5. GRPO [Group Relative Policy Optimization] uses these scores to update the model:
#    - Generates 4 responses [rollout.n=4] for each question
#    - Compares scores within the group: responses better than average get positive advantage [pushed up]
#    - Responses worse than average get negative advantage [pushed down]
#    - No separate critic model needed [unlike PPO] -- the group comparison IS the baseline

## 12. Cleanup

Run in order. Uncomment and run one cell at a time.

**Important:** Deleting the server does NOT delete the Cinder volume [the persistent disk with all your data]. The volume stays until you explicitly delete it. So your checkpoints and MLflow data are safe even after server deletion.

In [ ]:
# --- Check reservation status [is your GPU lease still active or expired?] ---
for lease_info in lease.list_leases():
    print(f"Lease: {lease_info.name} | Status: {lease_info.status} | Ends: {lease_info.end_date}")
    for r in lease_info.flavor_reservations:
        print(f"  flavor reservation: id={r['id']} status={r.get('status')}")
    for r in lease_info.node_reservations:
        print(f"  node reservation: id={r['id']} status={r.get('status')}")


In [ ]:
# --- Stop containers (run before deleting server) ---
# s.execute("cd /home/cc/toolrl && sudo docker compose down")
# print("Containers stopped.")

In [ ]:
# --- Delete the server [shuts down the VM, frees the GPU reservation] ---
# The Cinder volume [your persistent disk] is NOT deleted -- data stays safe.
# s.delete()
# print("Server deleted.")

In [ ]:
# --- Delete the lease [releases the GPU reservation back to the pool] ---
# Only do this when fully done -- once deleted you need a new lease to get GPUs again.
# l.delete()
# print("Lease deleted.")

In [ ]:
# --- Delete the Cinder volume [WARNING: permanently deletes all checkpoints and MLflow data] ---
# Only run this if you are completely done and don't need the data anymore.
# Volume must be detached first.

# Unmount and detach
# s.execute("sudo umount /mnt/block")
# s.detach_volume(volume.id)

# Then delete
# volume = cinder_client.volumes.get(volume.id)
# cinder_client.volumes.delete(volume)


In [ ]:
# Checkpoint to evaluate -- demo run saves to global_step_1
CHECKPOINT = "/app/models/toolrl-grpo-cold-qwen-1.5b-demo/actor/global_step_1"

# Step 1: Generate model outputs on the 597 API-Bank questions (~2-3 min)
print("Generating outputs...")
stdout, _ = s.execute(f"""
sudo docker exec verl bash -c '
    cd /workspace/benchmarks/API-Bank &&
    python3 generate_batch.py --model_paths {CHECKPOINT}
'
""")
print(stdout)

# Step 2: Score the outputs
print("Scoring...")
stdout, _ = s.execute(f"""
sudo docker exec verl bash -c '
    cd /workspace/benchmarks/API-Bank &&
    python3 evaluate.py --model_paths {CHECKPOINT}
'
""")
print(stdout)

## 11. Check Disk & Clean Old Checkpoints

Run before starting a new training run to avoid disk full errors.
Each 1.5B checkpoint is ~6-7 GB. The demo run saves one per step.

In [ ]:
# --- Check disk and existing checkpoints ---
stdout, _ = s.execute("df -h / | tail -1")
print("Disk:", stdout.strip())
stdout, _ = s.execute("sudo docker exec verl bash -c 'du -sh /app/models/toolrl-grpo-cold-qwen-1.5b-demo/actor/*/ 2>/dev/null | sort -V || echo empty'")
print("Checkpoints:\n", stdout)

# --- Kill any running training job (run before starting a new one) ---
# s.execute("sudo docker exec verl pkill -f 'verl.trainer.main_ppo'")
# print("Killed.")

# --- Delete ALL checkpoints (nuke everything, max disk freed) ---
# s.execute("sudo docker exec verl bash -c 'rm -rf /app/models/toolrl-grpo-cold-qwen-1.5b-demo && rm -f /tmp/train.log'")
# print("Cleaned all.")

# --- Delete specific checkpoints (keep only the ones you want) ---
# KEEP = {"global_step_1"}   # <-- set which steps to keep
# stdout, _ = s.execute("sudo docker exec verl bash -c 'ls /app/models/toolrl-grpo-cold-qwen-1.5b-demo/actor/'")
# all_steps = set(stdout.strip().split())
# to_delete = all_steps - KEEP
# for step in sorted(to_delete):
#     s.execute(f"sudo docker exec verl rm -rf /app/models/toolrl-grpo-cold-qwen-1.5b-demo/actor/{step}")
#     print(f"Deleted {step}")
# print(f"Kept: {KEEP}")

# --- Check training log (run anytime to see progress) ---
# stdout, _ = s.execute("sudo docker exec verl tail -20 /tmp/train.log 2>/dev/null || echo 'No log yet'")
# print(stdout)

# --- Check MLflow runs ---
# stdout, _ = s.execute("curl -s 'http://localhost:5000/api/2.0/mlflow/runs/search' -d '{\"experiment_ids\":[\"949894919383340008\"],\"max_results\":3,\"order_by\":[\"attribute.start_time DESC\"]}' -H 'Content-Type: application/json' | python3 -c 'import sys,json; [print(r[\"info\"][\"run_name\"],\"|\",r[\"info\"][\"status\"]) for r in json.load(sys.stdin)[\"runs\"]]'")
# print(stdout)

## 11. Cleanup

Run in order. Uncomment and run one cell at a time.

In [ ]:
# Stop containers
# s.execute("cd /home/cc/toolrl && sudo docker compose down")

In [ ]:
# Delete the server
# s.delete()

In [ ]:
# Delete the lease
# l.delete()
# print("Lease deleted.")

## (Optional) Resume from Existing Cinder Volume

Use this instead of Sections 1-8 if you already ran a full setup before and your Cinder volume still exists.

**What this does:** boots a new VM directly from your existing 512GB Cinder volume. Docker, the image, containers, checkpoints, HuggingFace cache, and MLflow data are all already there -- no reinstall needed.

**When to use:**
- Your previous reservation expired but you kept the Cinder volume
- You want to avoid the ~20 min Docker build + model download

**Requirements:**
- You know your Cinder volume ID (e.g. `067b7d0d-...`) -- find it in the KVM@TACC dashboard under Volumes
- You have an active lease with a GPU flavor reserved
- Run the config cell (Section 0) first

In [ ]:
# --- Resume from existing Cinder volume (replaces Sections 1-8 if you already have a volume) ---
# Fill in your volume ID, then uncomment and run.

# VOLUME_NAME = "projXX-toolrl-persist"   # match the name you used in Section 1c

# --- Step 1: Find the existing lease (you should still have an active lease from Section 1a) ---
# l = lease.get_lease(f"{PROJECT_ID}-toolrl-lease")
# l.show()

# --- Step 2: Get the existing volume ---
# cinder_client = chi.clients.cinder()
# volume = [v for v in cinder_client.volumes.list() if v.name == VOLUME_NAME][0]
# print(f"Resuming from volume: {volume.name} (id={volume.id})")

# --- Step 3: Launch server using the lease (re-uses Section 1b) ---
# s = server.Server(
#     f"{PROJECT_ID}-toolrl-server",
#     image_name=IMAGE_NAME,
#     flavor_name=l.get_reserved_flavors()[0].name,
# )
# s.submit(idempotent=True)
# s.associate_floating_ip()
# s.refresh()
# s.check_connectivity()

# --- Step 4: Re-attach + mount the volume (Section 2b) ---
# (Run the cell in Section 2b -- it is idempotent and will skip format if a filesystem already exists.)

# --- Step 5: Skip Sections 3-8 entirely -- the Docker image, models, datasets, and MLflow data
# are already on the volume from the previous lease. Jump straight to Section 9 (Run Training)
# or Section 11 (Check Disk).
